In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from etc.parse_ids import XMLParser

# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
PilotDC9 = data_folder / "PilotStudy_Afekta"

graph_gml = data_folder / "generated" / "modified_graph.gml"
human1_xml = data_folder / "Human-GEM.xml"

In [2]:
G = nx.read_gml(graph_gml)

In [3]:
chebi_match = pd.read_csv(PilotDC9 / "AFEKTA_chebis.csv")

In [4]:
name_match = pd.read_excel(PilotDC9 / "Supplementary_Material_2_Results_Dennisse_Avella.xlsx",sheet_name="keyIDs")

In [5]:
mask = name_match["Idlevel"] <= 2

# Clean empty values or '-' to None
curated = (
    name_match.loc[mask, "CuratedID"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
name_match.loc[mask, "CuratedID"] = curated.where(curated.notna(), None)
# Clean chebi_match
chebi_clean = chebi_match.copy()
chebi_clean["Input name"] = (
    chebi_clean["Input name"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
chebi_clean["CHEBI_ID"] = chebi_clean["CHEBI_ID"].replace({"": None, "-": None})
# Create lookup table from chebi_clean
lookup = (
    chebi_clean.dropna(subset=["Input name"])
    .drop_duplicates(subset=["Input name"], keep="first")
    .set_index("Input name")["CHEBI_ID"]
)
# Map Chebi_IDs to name_match
name_match.loc[mask, "Chebi_ID"] = name_match.loc[mask, "CuratedID"].map(lookup)

In [6]:
name_match

,Class,CuratedID,Idlevel,DatabaseID,Blood,Plasma,Mitra,Capitainer,Whatman,Chebi_ID
0,Amino acids and derivatives,1-methylhistidine,1.0,HMDB0000001,1,1,1,1,1,50599
1,Fatty acids and derivatives,2-Hydroxybutanoic acid,1.0,HMDB0000008 - HMDB0341410,1,1,1,1,1,1148
2,Other compounds,2-Piperidone,1.0,HMDB0011749,1,1,1,1,1,77761
3,Other compounds,3-Carboxy-4-methyl-5-propyl-2-furanpropionic acid,2.0,HMDB0061112,1,1,1,1,1,41254
4,Amino acids and derivatives,3-methylhistidine,2.0,HMDB0000479,1,1,1,1,1,27596
...,...,...,...,...,...,...,...,...,...,...
188,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
189,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
190,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
191,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN


In [2]:
parser = XMLParser(human1_xml)

In [3]:
df = parser.extract_data()

In [4]:
parser._first_of(df.Identifiers[68], {'kegg.compound'})

In [5]:
df.Identifiers[68]

[['bigg.metabolite', 'nrvnccrn'],
 ['vmhmetabolite', 'nrvnccrn'],
 ['metanetx.chemical', 'MNXM8942'],
 ['inchi',
  'InChI=1S',
  'C31H59NO4',
  'c1-5-6-7-8-9-10-11-12-13-14-15-16-17-18-19-20-21-22-23-24-25-26-31(35)36-29(27-30(33)34)28-32(2,3)4',
  'h12-13,29H,5-11,14-28H2,1-4H3',
  't29-',
  'm0',
  's1']]

In [11]:
df_human1 = parser.get_chebi_numbers()

In [6]:
df_human1 = parser.to_identifier_df()

C00964
C00964
C09880
C09880
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
C16525
C16525
C16525
C16525
C16180
C16180
C16180
C16522
C16522
C16522
C16522
None
None
None
C16179
C16179
C16179
C04654
None
None
C16531
C16531
C16531
C16531
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
C16533
C16533
C16533
C16533
None
None
None
C16645
C16645
C16645
None
None
C16532
C16532
C16532
None
None
None
None
C19559
C14791
C14786
C14786
C14792
C14787
C07712
C07712
C07712
C05450
C05448
C05754
C05275
C05275
None
None
None
C05758
C03221
C03221
None
None
None
None
None
None
None
None
None
None
None
C05763
C05272
C05272
C05748
C05271
C05271
None
None
None
None
C16218
C16218
C16218
C05751
C05276
C05276
None
None
None
None
None
None
C05760
C05273
C05273
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
C01011
None
None
None
C02944
C02944
None
None
None
None
None
None
None
None
C16173
C16173
C16173
C16